# 09 - Top 3 Model Comparison

**Goal:** Take the top 3 model types from the baseline run (notebook 08) and sweep their architectural hyperparameters - especially **layer count** - to see how depth/width changes CV and test MCC.

**Output isolation:** Everything in this notebook writes under `top_3_results/` (separate from baseline `artifacts/`). In Colab the directory is mirrored to `Drive/ANN-Project/Top_3_results/`.

---

## 0 - Imports & setup

In [ ]:
import sys
from pathlib import Path

# ============================================================
#  CONFIG SECIMI - Tek satirla degistir
# ============================================================
CONFIG_NAME = 'x7_fwd5'   # Options: 'base', 'x7_fwd1', 'x7_fwd5', 'x7_fwd21'
# ============================================================

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    TOOLKIT_PATH = '/content/drive/MyDrive/ANN-Project/toolkit.py'
    exec(open(TOOLKIT_PATH).read())
    setup()
    ROOT = Path('/content/repo')
else:
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

import json
import shutil
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.run_experiment import main as run_main
from src.models.model_factory import normalize_model_name

CONFIG_PATH = ROOT / 'configs' / f'{CONFIG_NAME}.yaml'
RESULTS_DIR = ROOT / 'top_3_results'
RESULTS_DIR.mkdir(exist_ok=True)

print(f'ROOT        : {ROOT}')
print(f'CONFIG      : {CONFIG_PATH.name}')
print(f'RESULTS_DIR : {RESULTS_DIR}')

---
## 1 - Top 3 model tipini baseline raporundan al

Notebook 08'in son `baseline_bull_bear_report.json` çıktısından her modelin en iyi varyantını (CV MCC) çıkarır. Rapor yoksa hardcoded fallback kullanır.

In [ ]:
baseline_metrics_dir = ROOT / 'artifacts' / 'metrics'
baseline_reports = sorted(baseline_metrics_dir.glob('*_report.json'))

FALLBACK_TOP_3 = [
    {'model': 'gru',                 'scaler': 'robust',   'quantile': 0.20, 'lookback': 42, 'cv_mcc': None},
    {'model': 'cnn1d',               'scaler': 'standard', 'quantile': 0.40, 'lookback': 42, 'cv_mcc': None},
    {'model': 'transformer_encoder', 'scaler': 'robust',   'quantile': 0.20, 'lookback': 5,  'cv_mcc': None},
]

if not baseline_reports:
    print('Baseline raporu bulunamadi. Hardcoded top 3 kullanilacak.')
    TOP_3 = FALLBACK_TOP_3
else:
    with open(baseline_reports[-1], encoding='utf-8') as f:
        baseline = json.load(f)

    rows = []
    for c in baseline.get('cv_candidates', []):
        rows.append({
            'model'   : normalize_model_name(c.get('model_name')),
            'scaler'  : c.get('scaler_name'),
            'quantile': c.get('threshold_quantile'),
            'lookback': c.get('lookback'),
            'cv_mcc'  : c.get('cv_summary', {}).get('mcc_mean'),
        })

    cand_df = pd.DataFrame(rows).dropna(subset=['cv_mcc'])
    best_per_model = (
        cand_df.sort_values('cv_mcc', ascending=False)
               .drop_duplicates('model', keep='first')
               .head(3)
               .reset_index(drop=True)
    )
    TOP_3 = best_per_model.to_dict('records')

print('Top 3 model (sweep edilecek):')
for entry in TOP_3:
    cv = entry.get('cv_mcc')
    cv_str = f'{cv:.4f}' if cv is not None else 'N/A'
    print(f"  {entry['model']:<22} scaler={entry['scaler']:<8} q={entry['quantile']:<5} lb={entry['lookback']:<3} baseline_cv_mcc={cv_str}")

---
## 2 - Mimari sweep tanimi

Her model icin **katman sayisi** ile birlikte ana boyut parametreleri (`hidden_dim`, `conv_channels`, `d_model`) varyasyonlu olarak verilir. Sabit (quantile, lookback, scaler) baseline'dan geliyor; tek degisken mimari.

| Model | Degisen parametreler |
|---|---|
| GRU | `hidden_dim`, `num_layers`, `dropout` |
| CNN1D | `conv_channels`, `kernel_size`, `num_conv_layers`, `dropout` |
| Transformer | `d_model`, `nhead`, `num_layers`, `dim_feedforward`, `dropout` |

Toplam 18 calistirma (her model 6 varyant). Her biri 4 fold CV + final test (~3-6 dk Colab GPU).

In [ ]:
ARCH_SWEEPS = {
    'gru': [
        {'hidden_dim': 32,  'num_layers': 1, 'dropout': 0.2},
        {'hidden_dim': 64,  'num_layers': 1, 'dropout': 0.2},
        {'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.2},
        {'hidden_dim': 64,  'num_layers': 2, 'dropout': 0.2},
        {'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.3},
        {'hidden_dim': 64,  'num_layers': 3, 'dropout': 0.3},
    ],
    'cnn1d': [
        {'conv_channels': 32,  'kernel_size': 3, 'num_conv_layers': 1, 'dropout': 0.2},
        {'conv_channels': 64,  'kernel_size': 3, 'num_conv_layers': 2, 'dropout': 0.2},
        {'conv_channels': 128, 'kernel_size': 3, 'num_conv_layers': 2, 'dropout': 0.2},
        {'conv_channels': 64,  'kernel_size': 5, 'num_conv_layers': 2, 'dropout': 0.2},
        {'conv_channels': 64,  'kernel_size': 5, 'num_conv_layers': 3, 'dropout': 0.3},
        {'conv_channels': 64,  'kernel_size': 7, 'num_conv_layers': 4, 'dropout': 0.3},
    ],
    'transformer_encoder': [
        {'d_model': 32,  'nhead': 4, 'num_layers': 1, 'dim_feedforward': 64,  'dropout': 0.2},
        {'d_model': 64,  'nhead': 4, 'num_layers': 1, 'dim_feedforward': 128, 'dropout': 0.2},
        {'d_model': 64,  'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.2},
        {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256, 'dropout': 0.2},
        {'d_model': 64,  'nhead': 4, 'num_layers': 4, 'dim_feedforward': 128, 'dropout': 0.3},
        {'d_model': 128, 'nhead': 8, 'num_layers': 4, 'dim_feedforward': 256, 'dropout': 0.3},
    ],
}

def variant_label(model, arch):
    if model == 'gru':
        return f"h{arch['hidden_dim']}_l{arch['num_layers']}_d{int(arch['dropout']*10)}"
    if model == 'cnn1d':
        return f"c{arch['conv_channels']}_k{arch['kernel_size']}_l{arch['num_conv_layers']}_d{int(arch['dropout']*10)}"
    if model == 'transformer_encoder':
        return f"d{arch['d_model']}_h{arch['nhead']}_l{arch['num_layers']}_d{int(arch['dropout']*10)}"
    return 'unknown'

total = sum(len(v) for v in ARCH_SWEEPS.values())
print(f'Toplam mimari varyant: {total}')
for model, archs in ARCH_SWEEPS.items():
    print(f'  {model:<22} : {len(archs)} varyant')

---
## 3 - Sweep'leri calistir

Her varyant icin `run_experiment.main()` cagrilir. Sweep boyunca:

- `experiment.name` benzersiz olur (`top3_<model>_<label>`) - boylece raporlar birbirini ezmez
- `artifacts.root_dir = top_3_results/artifacts` - baseline'i kirletmez
- `models.enabled = [model]`, `preprocessing.scalers = [scaler]` - sadece o model + scaler
- `threshold_quantile_candidates` ve `lookback_candidates` tek degere sabitlenir (baseline'in seciminden)

In [ ]:
results = []

for base in TOP_3:
    model = base['model']
    scaler = base['scaler']
    quantile = float(base['quantile'])
    lookback = int(base['lookback'])

    if model not in ARCH_SWEEPS:
        print(f'[SKIP] {model} icin ARCH_SWEEPS tanimi yok.')
        continue

    for arch in ARCH_SWEEPS[model]:
        label = variant_label(model, arch)
        exp_name = f'top3_{model}_{label}'

        overrides = {
            'experiment': {'name': exp_name},
            'models': {'enabled': [model]},
            'preprocessing': {'scalers': [scaler]},
            'labeling': {
                'threshold_quantile': quantile,
                'threshold_quantile_candidates': [quantile],
            },
            'sequence': {
                'lookback': lookback,
                'lookback_candidates': [lookback],
            },
            'artifacts': {'root_dir': 'top_3_results/artifacts'},
            model: arch,
        }

        print(f'\n=== {exp_name} ===')
        print(f'    arch = {arch}')
        try:
            report = run_main(str(CONFIG_PATH), config_overrides=overrides)
            cv_summary = report['best_cv_selection']['cv_summary']
            test_metrics = report['final_test']['metrics']
            row = {
                'model'       : model,
                'variant'     : label,
                'arch'        : str(arch),
                'cv_mcc'      : cv_summary.get('mcc_mean'),
                'cv_mcc_std'  : cv_summary.get('mcc_std'),
                'cv_f1'       : cv_summary.get('f1_mean'),
                'cv_bal_acc'  : cv_summary.get('balanced_accuracy_mean'),
                'test_mcc'    : test_metrics.get('mcc'),
                'test_f1'     : test_metrics.get('f1'),
                'test_bal_acc': test_metrics.get('balanced_accuracy'),
                'test_pr_auc' : test_metrics.get('pr_auc'),
            }
            results.append(row)
            print(f'    OK  CV_MCC={row["cv_mcc"]:.4f}  TEST_MCC={row["test_mcc"]:.4f}')
        except Exception as exc:
            print(f'    FAIL  {type(exc).__name__}: {exc}')
            traceback.print_exc()
            results.append({
                'model': model, 'variant': label, 'arch': str(arch),
                'error': f'{type(exc).__name__}: {exc}',
            })

results_df = pd.DataFrame(results)
results_csv = RESULTS_DIR / 'sweep_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'\nTamamlandi: {len(results)} run. Kaydedildi: {results_csv}')

---
## 4 - Her model kendi icinde siralama (CV MCC)

Her model tipinin kendi 6 varyanti, en yuksek CV MCC ustte.

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:.4f}' if pd.notna(v) else 'N/A')

show_cols = ['variant', 'cv_mcc', 'cv_mcc_std', 'test_mcc', 'test_f1', 'test_bal_acc', 'arch']
successful = results_df.dropna(subset=['cv_mcc']) if 'cv_mcc' in results_df.columns else results_df

for model in successful['model'].unique():
    sub = (
        successful[successful['model'] == model]
            .sort_values('cv_mcc', ascending=False)
            .reset_index(drop=True)
    )
    print(f'\n=== {model} ===')
    print(sub[show_cols].to_string())

---
## 5 - Genel siralama (tum varyantlar)

18 varyant tek tabloda, CV MCC azalan.

In [ ]:
overall = (
    successful
        .sort_values('cv_mcc', ascending=False)
        .reset_index(drop=True)
        .assign(rank=lambda d: d.index + 1)
)

cols = ['rank', 'model', 'variant', 'cv_mcc', 'cv_mcc_std', 'test_mcc', 'test_f1', 'test_bal_acc', 'test_pr_auc']
print(overall[cols].to_string(index=False))

print('\n=== Her modelin en iyi varyanti (genel siralamada) ===')
best_per_model = overall.drop_duplicates('model', keep='first').reset_index(drop=True)
print(best_per_model[cols].to_string(index=False))

---
## 6 - Gorsellestirme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_per_model_sorted = best_per_model.sort_values('cv_mcc', ascending=True)
axes[0].barh(
    best_per_model_sorted['model'] + ' / ' + best_per_model_sorted['variant'],
    best_per_model_sorted['cv_mcc'],
    color='steelblue', edgecolor='black',
)
axes[0].set_xlabel('Best CV MCC')
axes[0].set_title('Her Model Tipinin En Iyi Varyanti', fontsize=11)
axes[0].axvline(0, color='black', linewidth=0.8)

model_palette = {'gru': '#1f77b4', 'cnn1d': '#2ca02c', 'transformer_encoder': '#d62728'}
for model in successful['model'].unique():
    sub = successful[successful['model'] == model].dropna(subset=['cv_mcc', 'test_mcc'])
    axes[1].scatter(
        sub['cv_mcc'], sub['test_mcc'],
        label=model, s=80, alpha=0.75,
        color=model_palette.get(model, None), edgecolor='black',
    )
if len(successful) > 0:
    cv_vals = successful['cv_mcc'].dropna()
    test_vals = successful['test_mcc'].dropna()
    if len(cv_vals) > 0 and len(test_vals) > 0:
        lo = min(cv_vals.min(), test_vals.min()) - 0.02
        hi = max(cv_vals.max(), test_vals.max()) + 0.02
        axes[1].plot([lo, hi], [lo, hi], 'k--', alpha=0.3, label='y=x')
axes[1].set_xlabel('CV MCC')
axes[1].set_ylabel('Test MCC')
axes[1].set_title('CV vs Test MCC (her nokta = 1 varyant)', fontsize=11)
axes[1].axhline(0, color='black', linewidth=0.5, alpha=0.5)
axes[1].axvline(0, color='black', linewidth=0.5, alpha=0.5)
axes[1].legend()

plt.tight_layout()
fig_path = RESULTS_DIR / 'top3_comparison.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Sekil: {fig_path}')

---
## 7 - Loss & val_loss egrileri

Her sweep variant'in 4 fold'unda `train_loss` ve `val_loss` epoch bazli kaydedilir (sadece torch modelleri icin - sklearn modellerinde tek-sayilik training history vardir).

Bu bolum:
1. **Best-per-model** uc varyant icin (overall siralamadan) 4 fold subplot
2. **`plot_loss(model, variant)`** yardimci fonksiyonu - istedigin varyanti tek satirla cizmek icin

Egriler `top_3_results/loss_<exp_name>.png` olarak da kaydedilir.

In [ ]:
metrics_dir = RESULTS_DIR / 'artifacts' / 'metrics'

def load_fold_history(exp_name):
    """exp_name = 'top3_gru_h64_l1_d2' gibi. Folds JSON'unu bulup fold_results listesini doner."""
    matches = sorted(metrics_dir.glob(f'{exp_name}_*_folds.json'))
    if not matches:
        return None, None
    folds_path = matches[0]
    with open(folds_path, encoding='utf-8') as f:
        data = json.load(f)
    return data.get('fold_results', []), folds_path

def plot_loss_for_experiment(exp_name, show=True):
    folds, folds_path = load_fold_history(exp_name)
    if folds is None:
        print(f'  [SKIP] {exp_name}: folds JSON bulunamadi.')
        return
    n_folds = len(folds)
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 3.5), sharey=True)
    if n_folds == 1:
        axes = [axes]

    for i, fold in enumerate(folds):
        trainer_info = fold.get('trainer_info', {})
        history = trainer_info.get('history', [])
        ax = axes[i]
        if not history:
            ax.text(0.5, 0.5, 'no history\n(sklearn?)', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='gray')
            ax.set_title(f"Fold {fold.get('fold_index', i)}")
            continue
        epochs = [h.get('epoch') for h in history]
        train_loss = [h.get('train_loss') for h in history]
        val_loss = [h.get('val_loss') for h in history]
        ax.plot(epochs, train_loss, label='train_loss', color='steelblue', linewidth=1.5)
        ax.plot(epochs, val_loss, label='val_loss', color='darkorange', linewidth=1.5)
        best_epoch = trainer_info.get('best_epoch')
        if best_epoch is not None:
            ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.6,
                       label=f'best ep {best_epoch}')
        ax.set_title(f"Fold {fold.get('fold_index', i)}", fontsize=10)
        ax.set_xlabel('epoch')
        if i == 0:
            ax.set_ylabel('loss')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    fig.suptitle(exp_name, fontsize=12, y=1.02)
    plt.tight_layout()
    save_path = RESULTS_DIR / f'loss_{exp_name}.png'
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f'  Kaydedildi: {save_path}')

def plot_loss(model, variant):
    """Kisayol: plot_loss('gru', 'h64_l1_d2')"""
    plot_loss_for_experiment(f'top3_{model}_{variant}')

# --- Best-per-model uc varyant ---
print('=== Best-per-model loss egrileri ===')
for _, row in best_per_model.iterrows():
    exp_name = f"top3_{row['model']}_{row['variant']}"
    print(f'\n--- {exp_name} (CV MCC={row["cv_mcc"]:.4f}, Test MCC={row["test_mcc"]:.4f}) ---')
    plot_loss_for_experiment(exp_name)

print('\nIstedigin varyanti tek satirla cizmek icin:')
print("  plot_loss('gru', 'h64_l2_d2')")
print("  plot_loss('cnn1d', 'c128_k3_l2_d2')")
print("  plot_loss('transformer_encoder', 'd128_h4_l2_d2')")

---
## 8 - Drive'a kaydet (Colab)

Tum `top_3_results/` icerigi (artifacts + sweep_results.csv + top3_comparison.png + loss_*.png) `Drive/ANN-Project/Top_3_results/` altina kopyalanir.

In [ ]:
if IN_COLAB:
    drive_dest = Path('/content/drive/MyDrive/ANN-Project/Top_3_results')
    drive_dest.mkdir(parents=True, exist_ok=True)

    for item in RESULTS_DIR.iterdir():
        dest = drive_dest / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

    print(f'>> Drive: {drive_dest}')
    for p in sorted(drive_dest.rglob('*')):
        if p.is_file():
            print(f'   {p.relative_to(drive_dest)}')
else:
    print(f'Local mode - sonuclar: {RESULTS_DIR}')